### CI2604P Learning Log — Smoker Status Prediction


### 1.Project Overview

---

**Objective: 
Predict smoking status (1 = smoker, 0 = non-smoker) from physical, cardiovascular, and laboratory bio-signal measurements.
Evaluation Metric used : ROC AUC (Receiver Operating Characteristic Area Under the Curve). 
The final output must be predicted probabilities (predict_proba) rather than hard binary class labels.**



|train.csv|
|---|
   (15,000 rows, 24 columns) 
 
|test.csv|
|---|
   (10,000 rows, 23 columns).


   ---

### Data Understanding & Cleaning:


Class Distribution: Slightly imbalanced (63.42% non-smokers vs. 36.58% smokers).


**Data quality checks**

- **Missing values:** none in either train or test — no imputation required.

- **Duplicate rows:** none found in train.

- Ran `.describe()` on all numeric columns to check ranges, means, and spot anything implausible (e.g. negative values, impossible measurements).
**Cleaning issues found and handled**
- **Sentinel value in eyesight columns:** the dataset used `9.9` to represent "blind," which would distort an averaged eyesight feature if left as-is. Clipped `eyesight(left)` and `eyesight(right)` to a maximum of 2.5 before using them.
- **Skewed lab values:** `triglyceride`, `Gtp`, `AST`, `ALT`, and `LDL` showed heavy right-skew and outliers on boxplots. Rather than removing these rows (which could throw away real signal from unhealthy/smoker cases), they were kept and log-transformed later during feature engineering.



### 2. Feature Engineering


**Created 16 engineered features**

| Theme | Features created |
|---|---|
Body composition:	              |  BMI, waist_height_ratio
Blood pressure:	                  | pulse_pressure, mean_arterial_pressure
Eyesight/hearing:	              |  eyesight_avg (with sentinel value capped), hearing_any_loss
Lipid panel:	                  |  chol_hdl_ratio, ldl_hdl_ratio, triglyceride_hdl_ratio
Liver enzymes:	                  |  ast_alt_ratio (De Ritis ratio)
Skew correction:	              |  log_triglyceride, log_Gtp, log_AST, log_ALT, log_LDL, log_serum creatinine

---
## 3. Preprocessing
 
- Split into 38 features after engineering (from the original 22 raw predictors).

- Used an 80/20 stratified train/validation split to preserve class balance.

- Applied `RobustScaler` (robust to outliers) — but only for the linear model, since tree-based models don't need scaled inputs.


## 4.Model Evaluation

- Evaluated: Tested four machine learning models using 5-fold Stratified Cross-Validation:

- Logistic Regression: Baseline linear approach (required RobustScaler).

- Random Forest / LightGBM / XGBoost: Tree-based ensemble methods. 

- XGBoost and tuned LightGBM yielded the highest validation scores (~0.89 ROC AUC).



 ---

- The evaluation metric is ROC AUC, which ranks predictions based on confidence rather than a strict 0 or 1 cutoff. 
Using predict_proba preserves the model's exact certainty for each patient, maximizing the evaluation score.

- Summary of the issues encountered and how it was addressed:Issue: Heavy skewness in raw lab metrics weakened the baseline linear model performance.

- Solution: Engineered medical ratio features and log-scaled highly skewed inputs, which noticeably boosted model prediction accuracy across all cross-validation folds.

--- 

## 5. Model Comparison
 
Four models were compared with 5-fold stratified cross-validation:
 
| Model | CV AUC | Val AUC |
|---|---|---|
| Logistic Regression | 0.8768 | 0.8764 |
| Random Forest | 0.8835 | 0.8818 |
| **XGBoost** | **0.8876** | **0.8902** |
| LightGBM | 0.8862 | 0.8856 |

### Trade-offs

| Model | CV AUC | Val AUC | Strengths | Weaknesses |
|---|---|---|---|---|
| **Logistic Regression** | 0.8768 | 0.8764 | Fast to train; simple and interpretable (coefficients show direction/size of each feature's effect); low variance, unlikely to overfit; good baseline | Assumes linear relationships, so it can't capture interactions or non-linear effects between features without manual engineering; needed feature scaling (`RobustScaler`), adding a preprocessing step the tree models didn't require; lowest AUC of the four |
| **Random Forest** | 0.8835 | 0.8818 | Captures non-linear relationships and feature interactions automatically; robust to outliers and doesn't need feature scaling; less prone to overfitting than a single decision tree due to bagging | Slower to train and predict than boosting methods at similar performance; larger memory footprint (many deep trees); less interpretable than linear models; underperformed both boosting models here |
| **XGBoost** | 0.8876 | 0.8902 | Best overall performance (highest val AUC); handles non-linearities and interactions well; regularization (L1/L2) helps control overfitting; efficient with `n_jobs=-1` parallelism |More hyperparameters to tune than Random Forest; slower to train than LightGBM on large data; can overfit if not tuned carefully.
| **LightGBM** | 0.8862 | 0.8856 | Fastest training among the boosting models (leaf-wise growth, histogram-based splits); scales well to larger datasets; strong performance close to XGBoost; became the model selected for tuning | Leaf-wise growth can overfit on smaller datasets if `num_leaves`/`min_child_samples` aren't controlled; slightly more sensitive to hyperparameter choices than XGBoost; marginally lower val AUC than XGBoost in the baseline comparison |

### 6.Hyperparameter Tuning

Tuned LightGBM using RandomizedSearchCV (25 iterations) over 9 hyperparameters 
(tree depth, leaves, learning rate, subsampling, regularization, etc.).



Best CV ROC AUC: 0.8892, improving to a held-out validation AUC of 0.8899 — slightly better than the untuned baseline models.
Plotted the validation ROC curve and inspected the top 15 feature importances.

### 7.Roadblocks & Debugging:

- **Sentinel values in eyesight data:** The dataset used `9.9` as a sentinel for "blind," which would have badly skewed the `eyesight_avg` feature if left uncapped. Caught this during EDA and clipped eyesight values to a maximum of 2.5 before averaging.
- **Divide-by-zero risk in ratio features:** Several engineered features involved division (e.g. `ast_alt_ratio = AST / ALT`, cholesterol/HDL ratios). Some lab values could be zero, which would produce `inf` or `NaN`. Handled this by replacing zero denominators with `NaN` (e.g. `ALT.replace(0, np.nan)`) before dividing.
- **Right-skewed lab values:** Columns like `triglyceride`, `Gtp`, `AST`, `ALT`, and `LDL` had heavy right-skew and visible outliers (confirmed via boxplots). Log-transforming these (`log_triglyceride`, `log_Gtp`, etc.) rather than dropping or capping outliers, to avoid discarding potentially informative extreme values.
---

## 8. Final Training & Submission

- Retrained the tuned LightGBM model on **100% of the training data** (not just the 80% split) before generating predictions, to use all available signal.
- Noted the in-sample AUC (0.9172) is optimistic and **not** the number to trust — the held-out validation AUC (0.8899) is the honest estimate of generalization performance.
- Generated predicted probabilities (not hard labels, since the competition scores on ROC AUC) for the test set.
- Ran sanity checks before saving: probabilities in [0,1], correct row count, unique IDs.
- Saved `submission.csv` — final predictions ranged roughly from ~0.54 to ~0.85 in probability of being a smoker for the first few rows.